# Search-7-MCTS-And-Beyond : Monte Carlo Tree Search et Extensions

**Navigation** : [<< Recherche adversariale](Search-6-AdversarialSearch.ipynb) | [Index](../README.md) | [Dancing Links >>](Search-8-DancingLinks.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Comprendre** les limites de Minimax et pourquoi MCTS a revolutionne les jeux
2. **Implementer** l'algorithme MCTS avec UCB1
3. **Comparer** MCTS vs Minimax sur differents types de jeux
4. **Decouvrir** OpenSpiel, le framework de Google pour les jeux
5. **Explorer** les approches hybrides (AlphaGo, AlphaZero)

### Prerequis
- Notebook Search-6 (AdversarialSearch, Minimax, Alpha-Beta)
- Bases de probabilites et statistiques
- Bases de Python : classes, recursion

### Duree estimee : 90 minutes

In [ ]:
# Imports
import sys
import time
import random
import math
from typing import Optional, List, Dict, Tuple, Any
from copy import deepcopy
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

print("Environnement pret pour MCTS.")

## 1. Limites de Minimax

### Pourquoi Minimax ne suffit pas

L'algorithme Minimax avec Alpha-Beta fonctionne bien pour les jeux simples, mais rencontre des limites :

1. **Explosion combinatoire** : Aux echecs, meme avec Alpha-Beta, on ne peut explorer que 6-8 demi-coups en profondeur
2. **Fonction d'evaluation** : Necessite une expertise humaine pour concevoir une bonne heuristique
3. **Horizon effect** : Des evenements importants au-dela de la profondeur sont ignores

### La revolution AlphaGo (2016)

AlphaGo a battu Lee Sedol (champion du monde de Go) en utilisant :
- **MCTS** pour la recherche
- **Reseaux de neurones** pour l'evaluation (policy + value networks)

MCTS permet d'explorer intelligemment sans fonction d'evaluation experte !

## 2. L'Algorithme MCTS

### Principe

**Monte Carlo Tree Search** construit progressivement un arbre de recherche en :
1. **Selection** : Traverser l'arbre avec UCB1 pour equilibrer exploration/exploitation
2. **Expansion** : Ajouter un nouveau noeud a l'arbre
3. **Simulation** : Jouer une partie aleatoire jusqu'a la fin (rollout)
4. **Backpropagation** : Remonter le resultat dans l'arbre

### UCB1 (Upper Confidence Bound)

UCB1 equilibre exploration et exploitation :

```
UCB1 = W/N + c * sqrt(ln(N_parent) / N)
```

- **W/N** : Taux de victoire (exploitation)
- **c * sqrt(...)** : Terme d'exploration (c = 1.41 typiquement)
- **N** : Nombre de visites du noeud
- **N_parent** : Nombre de visites du parent

In [ ]:
class NoeudMCTS:
    """Noeud de l'arbre MCTS."""
    
    def __init__(self, etat: Any, parent=None, action=None):
        self.etat = etat
        self.parent = parent
        self.action = action  # Action qui a mene a cet etat
        self.enfants = {}     # action -> NoeudMCTS
        self.visites = 0
        self.victoires = 0.0
        self.actions_non_explorees = None  # Initialise lors de la premiere expansion
    
    def ucb1(self, c: float = 1.41) -> float:
        """Calcul du score UCB1."""
        if self.visites == 0:
            return float('inf')
        
        exploitation = self.victoires / self.visites
        exploration = c * math.sqrt(math.log(self.parent.visites) / self.visites)
        return exploitation + exploration
    
    def meilleur_enfant_ucb1(self, c: float = 1.41) -> 'NoeudMCTS':
        """Selectionne l'enfant avec le meilleur score UCB1."""
        return max(self.enfants.values(), key=lambda n: n.ucb1(c))
    
    def meilleur_enfant_visites(self) -> 'NoeudMCTS':
        """Selectionne l'enfant le plus visite (pour le coup final)."""
        return max(self.enfants.values(), key=lambda n: n.visites)
    
    def est_fully_expanded(self, jeu) -> bool:
        """Verifie si toutes les actions ont ete expandues."""
        if self.actions_non_explorees is None:
            self.actions_non_explorees = list(jeu.actions(self.etat))
        return len(self.actions_non_explorees) == 0
    
    def est_terminal(self, jeu) -> bool:
        return jeu.est_terminal(self.etat)

In [ ]:
class MCTS:
    """Implementation de Monte Carlo Tree Search."""
    
    def __init__(self, jeu, c: float = 1.41):
        self.jeu = jeu
        self.c = c  # Parametre d'exploration
        self.stats = {'selections': 0, 'expansions': 0, 'simulations': 0, 'backprops': 0}
    
    def recherche(self, etat: Any, iterations: int = 1000) -> Tuple[Any, float]:
        """
        Execute MCTS depuis l'etat donne.
        Retourne (meilleure_action, valeur_estimee).
        """
        racine = NoeudMCTS(etat)
        
        for _ in range(iterations):
            noeud = self._selection(racine)
            resultat = self._simulation(noeud)
            self._backpropagation(noeud, resultat)
        
        meilleur = racine.meilleur_enfant_visites()
        valeur = meilleur.victoires / meilleur.visites if meilleur.visites > 0 else 0
        return meilleur.action, valeur
    
    def _selection(self, noeud: NoeudMCTS) -> NoeudMCTS:
        """Descend dans l'arbre jusqu'a trouver un noeud a explorer."""
        self.stats['selections'] += 1
        
        while not noeud.est_terminal(self.jeu):
            if not noeud.est_fully_expanded(self.jeu):
                return self._expansion(noeud)
            else:
                noeud = noeud.meilleur_enfant_ucb1(self.c)
        
        return noeud
    
    def _expansion(self, noeud: NoeudMCTS) -> NoeudMCTS:
        """Ajoute un nouvel enfant a l'arbre."""
        self.stats['expansions'] += 1
        
        if noeud.actions_non_explorees is None:
            noeud.actions_non_explorees = list(self.jeu.actions(noeud.etat))
        
        action = noeud.actions_non_explorees.pop()
        nouvel_etat = self.jeu.resultat(noeud.etat, action)
        enfant = NoeudMCTS(nouvel_etat, parent=noeud, action=action)
        noeud.enfants[action] = enfant
        return enfant
    
    def _simulation(self, noeud: NoeudMCTS) -> float:
        """
        Simule une partie aleatoire depuis le noeud.
        Retourne le resultat du point de vue du joueur MAX.
        """
        self.stats['simulations'] += 1
        
        etat = noeud.etat
        joueur_max_original = self.jeu.joueur(noeud.parent.etat) if noeud.parent else 'MAX'
        
        while not self.jeu.est_terminal(etat):
            actions = list(self.jeu.actions(etat))
            action = random.choice(actions)
            etat = self.jeu.resultat(etat, action)
        
        return self.jeu.utilite(etat, joueur_max_original)
    
    def _backpropagation(self, noeud: NoeudMCTS, resultat: float):
        """Remonte le resultat dans l'arbre."""
        self.stats['backprops'] += 1
        
        while noeud is not None:
            noeud.visites += 1
            
            # Le resultat depend du point de vue du joueur a ce noeud
            if self.jeu.joueur(noeud.etat) == 'MAX':
                noeud.victoires += resultat
            else:
                noeud.victoires += -resultat  # Inverser pour MIN
            
            noeud = noeud.parent

## 3. Test sur Tic-Tac-Toe

Comparons MCTS avec Minimax sur le jeu de Morpion.

In [ ]:
# Reutilisation de la classe TicTacToe du notebook precedent
from abc import ABC, abstractmethod

class JeuSommeNulle(ABC):
    @abstractmethod
    def etat_initial(self) -> Any: pass
    @abstractmethod
    def joueur(self, etat: Any) -> str: pass
    @abstractmethod
    def actions(self, etat: Any) -> List[Any]: pass
    @abstractmethod
    def resultat(self, etat: Any, action: Any) -> Any: pass
    @abstractmethod
    def est_terminal(self, etat: Any) -> bool: pass
    @abstractmethod
    def utilite(self, etat: Any, joueur: str) -> float: pass

class TicTacToe(JeuSommeNulle):
    def __init__(self):
        self._etat_initial = (tuple([' ']*9), 'X')
    
    def etat_initial(self):
        return self._etat_initial
    
    def joueur(self, etat):
        return 'MAX' if etat[1] == 'X' else 'MIN'
    
    def actions(self, etat):
        return [i for i in range(9) if etat[0][i] == ' ']
    
    def resultat(self, etat, action):
        grille = list(etat[0])
        joueur = etat[1]
        grille[action] = joueur
        prochain = 'O' if joueur == 'X' else 'X'
        return (tuple(grille), prochain)
    
    def est_terminal(self, etat):
        grille = etat[0]
        lignes = [[0,1,2],[3,4,5],[6,7,8],[0,3,6],[1,4,7],[2,5,8],[0,4,8],[2,4,6]]
        for l in lignes:
            if grille[l[0]] != ' ' and grille[l[0]] == grille[l[1]] == grille[l[2]]:
                return True
        return ' ' not in grille
    
    def utilite(self, etat, joueur):
        grille = etat[0]
        lignes = [[0,1,2],[3,4,5],[6,7,8],[0,3,6],[1,4,7],[2,5,8],[0,4,8],[2,4,6]]
        for l in lignes:
            if grille[l[0]] != ' ' and grille[l[0]] == grille[l[1]] == grille[l[2]]:
                gagnant = 'MAX' if grille[l[0]] == 'X' else 'MIN'
                return 1 if gagnant == joueur else -1
        return 0

# Test MCTS
jeu = TicTacToe()
mcts = MCTS(jeu)

start = time.time()
action, valeur = mcts.recherche(jeu.etat_initial(), iterations=1000)
temps = time.time() - start

print(f"MCTS (1000 iterations): action={action}, valeur={valeur:.3f}, temps={temps:.3f}s")
print(f"Stats: {mcts.stats}")

## 4. Comparaison MCTS vs Minimax

Comparons les deux approches sur differents criteres.

In [ ]:
# Minimax pour comparaison
def minimax(jeu, etat, joueur_max='MAX'):
    if jeu.est_terminal(etat):
        return jeu.utilite(etat, joueur_max), None
    
    actions = jeu.actions(etat)
    if jeu.joueur(etat) == joueur_max:
        best_v, best_a = float('-inf'), None
        for a in actions:
            v, _ = minimax(jeu, jeu.resultat(etat, a), joueur_max)
            if v > best_v:
                best_v, best_a = v, a
        return best_v, best_a
    else:
        best_v, best_a = float('+inf'), None
        for a in actions:
            v, _ = minimax(jeu, jeu.resultat(etat, a), joueur_max)
            if v < best_v:
                best_v, best_a = v, a
        return best_v, best_a

# Benchmark
resultats = []

# Minimax
start = time.time()
v_mm, a_mm = minimax(jeu, jeu.etat_initial())
t_mm = time.time() - start
resultats.append(('Minimax', t_mm, v_mm, a_mm))

# MCTS avec differents nombres d'iterations
for n_iter in [100, 500, 1000, 5000]:
    mcts = MCTS(jeu)
    start = time.time()
    v, a = mcts.recherche(jeu.etat_initial(), iterations=n_iter)
    t = time.time() - start
    resultats.append((f'MCTS ({n_iter})', t, v, a))

# Affichage
df = pd.DataFrame(resultats, columns=['Algorithme', 'Temps (s)', 'Valeur', 'Action'])
display(df)

### Analyse des resultats

| Critere | Minimax | MCTS |
|---------|---------|------|
| **Garantie d'optimalite** | Oui (si arbre complet) | Non (probabiliste) |
| **Fonction d'evaluation** | Necessaire | Non necessaire |
| **Complexite** | O(b^d) | O(n * d) ou n = iterations |
| **Parallellisable** | Difficile | Tres facile |
| **Jeux a grand facteur de branchement** | Impraticable | Adapté |

### Quand utiliser MCTS ?

- Jeux avec grand espace d'etat (Go, Hex)
- Jeux sans bonne fonction d'evaluation
- Situations avec contrainte de temps variable
- Jeux avec hasard (backgammon, poker)

## 5. OpenSpiel - Framework de Jeux

### Presentation

**OpenSpiel** est un framework open-source de DeepMind pour la recherche en IA sur les jeux.

### Caracteristiques

- **40+ jeux** : Echecs, Go, Hex, Poker, Hanabi, etc.
- **Multi-agent** : Jeux a N joueurs
- **Information imparfaite** : Poker, Hanabi
- **Algos inclus** : MCTS, AlphaZero, CFR, etc.

### Installation

```bash
# Linux/WSL uniquement
pip install open-spiel
```

In [ ]:
# Note: OpenSpiel necessite Linux/WSL
# Ce code est un exemple de l'API (ne fonctionnera pas sur Windows natif)

openspiel_available = False

try:
    import pyspiel
    openspiel_available = True
    
    # Charger le jeu de Tic-Tac-Toe
    game = pyspiel.load_game("tic_tac_toe")
    state = game.new_initial_state()
    
    print(f"Jeu: {game}")
    print(f"Etat initial: {state}")
    
    # MCTS d'OpenSpiel
    from open_spiel.python.algorithms import mcts
    
    # Configuration MCTS
    rng = np.random.RandomState(42)
    evaluator = mcts.RandomRolloutEvaluator(1, rng)
    bot = mcts.MCTSBot(game, 2.0, 1000, evaluator, rng)
    
    # Jouer un coup
    action = bot.step(state)
    print(f"Action MCTS: {action}")
    
except ImportError:
    print("OpenSpiel n'est pas installe ou non disponible sur cette plateforme.")
    print("Installation: pip install open-spiel (Linux/WSL uniquement)")

## 6. AlphaGo et AlphaZero

### Architecture AlphaGo (2016)

AlphaGo combine MCTS avec des reseaux de neurones :

1. **Policy Network** : Predit la probabilite de chaque coup
2. **Value Network** : Evalue la position
3. **MCTS** : Guide la recherche avec les predictions des reseaux

### AlphaZero (2017)

AlphaZero simplifie et generalise l'approche :

- **Auto-apprentissage** : Pas de donnees humaines
- **Reseau unique** : Policy + Value ensemble
- **Universel** : Fonctionne pour Go, Echecs, Shogi

### Principe de l'apprentissage

```
1. Initialiser le reseau aleatoirement
2. Jouer des parties contre soi-meme avec MCTS
3. Entrainer le reseau sur les positions et resultats
4. Repeter jusqu'a convergence
```

In [ ]:
# Sketch conceptuel d'un AlphaZero simplifie

class AlphaZeroMCTS:
    """
    MCTS guide par un reseau de neurones (version simplifiee).
    En pratique, utiliser des frameworks comme PyTorch ou TensorFlow.
    """
    
    def __init__(self, jeu, policy_value_fn, c=1.41):
        self.jeu = jeu
        self.policy_value_fn = policy_value_fn
        self.c = c
    
    def recherche(self, etat, iterations=100):
        """
        MCTS avec evaluation par reseau de neurones.
        policy_value_fn(etat) -> (probs_dict, value)
        """
        racine = NoeudMCTS(etat)
        
        for _ in range(iterations):
            noeud = racine
            chemin = [noeud]
            
            # Selection
            while noeud.enfants and noeud.est_fully_expanded(self.jeu):
                noeud = noeud.meilleur_enfant_ucb1(self.c)
                chemin.append(noeud)
            
            # Evaluation par le reseau
            if noeud.est_terminal(self.jeu):
                value = self.jeu.utilite(noeud.etat, 'MAX')
            else:
                probs, value = self.policy_value_fn(noeud.etat)
                
                # Expansion avec les probabilites du policy network
                for action in self.jeu.actions(noeud.etat):
                    enfant = NoeudMCTS(
                        self.jeu.resultat(noeud.etat, action),
                        parent=noeud,
                        action=action
                    )
                    noeud.enfants[action] = enfant
            
            # Backpropagation
            for n in chemin:
                n.visites += 1
                n.victoires += value
        
        return racine.meilleur_enfant_visites().action

def random_policy_value(etat):
    """Policy/Value network fictif (aleatoire) pour demonstration."""
    jeu = TicTacToe()
    actions = list(jeu.actions(etat))
    probs = {a: 1.0/len(actions) for a in actions}
    value = random.uniform(-0.5, 0.5)
    return probs, value

print("Sketch AlphaZero implemente (conceptuel).")

## 7. Extensions Avancees

### RAVE (Rapid Action Value Estimation)

RAVE utilise l'heuristique "all moves as first" pour accelerer l'apprentissage :
- Un coup est evalue meme s'il est joue plus tard dans la partie
- Acceleration significative dans les premiers stades

### UCT (UCB applied to Trees)

UCT est le nom original de l'algorithme MCTS avec UCB1. Variantes :
- **UCB1-Tuned** : Ajuste le parametre d'exploration
- **UCB-V** : Utilise la variance des recompenses

### Parallelisation

MCTS se parallellise facilement :
- **Leaf parallelisation** : Simulations en parallele
- **Root parallelisation** : Plusieurs arbres en parallele
- **Tree parallelisation** : Acces concurrent a l'arbre

In [ ]:
# Exemple de parallelisation simple avec multiprocessing
from multiprocessing import Pool

def mcts_worker(args):
    """Worker pour parallelisation."""
    jeu, etat, iterations = args
    mcts = MCTS(jeu)
    action, _ = mcts.recherche(etat, iterations)
    return action

def mcts_parallel(jeu, etat, total_iterations=1000, n_workers=4):
    """
    MCTS parallele avec root parallelisation.
    """
    iterations_per_worker = total_iterations // n_workers
    args = [(jeu, etat, iterations_per_worker) for _ in range(n_workers)]
    
    with Pool(n_workers) as pool:
        actions = pool.map(mcts_worker, args)
    
    # Vote majoritaire
    from collections import Counter
    vote = Counter(actions)
    return vote.most_common(1)[0][0]

print("MCTS parallele implemente (ne fonctionne pas dans un notebook interactif).")

## 8. Synthese

### Resume des approches

| Approche | Force | Faiblesse |
|----------|-------|-----------|
| **Minimax + Alpha-Beta** | Optimal si arbre complet | Explosion combinatoire |
| **MCTS pur** | Pas de fonction d'evaluation | Convergence lente |
| **MCTS + Heuristiques** | Rapide, bon compromis | Heuristiques necessaires |
| **AlphaZero** | Auto-apprentissage | Ressources enormes |

### Quand utiliser quoi ?

- **Jeux simples** (Tic-Tac-Toe, petits puzzles) : Minimax
- **Jeux moyens** (Connect Four, Othello) : Alpha-Beta + heuristiques
- **Jeux complexes** (Go, Hex) : MCTS + reseaux de neurones
- **Jeux a information imparfaite** (Poker, Hanabi) : CFR + deep learning

### Pour aller plus loin

- **MuZero** : Apprend les regles du jeu (model-based)
- **AlphaFold** : Applications hors jeux (biologie)
- **Expert Iteration** : Combinaison expert/apprenti

---

**Navigation** : [<< Recherche adversariale](Search-6-AdversarialSearch.ipynb) | [Index](../README.md) | [Dancing Links >>](Search-8-DancingLinks.ipynb)

## Exercices

1. **Parametre c** : Testez differentes valeurs de c (0.5, 1.0, 1.41, 2.0) et analysez l'impact
2. **Rollout intelligent** : Implementez un rollout avec heuristiques au lieu d'aleatoire
3. **Connect Four** : Comparez MCTS et Alpha-Beta sur Connect Four
4. **RAVE** : Implementez l'extension RAVE pour accelerer l'apprentissage
5. **OpenSpiel** : Explorez OpenSpiel et testez differents jeux